## **Preparando o ambiente para utilizar o Sparklyr**

### Instalando o Java

O Apache Spark depende de outros sistemas, portanto, antes do Spark é preciso instalar as dependências. Primeiro, deve-se instalar o java

----


In [ ]:
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)
  cat(paste0(result, collapse = "\n"))
}

shell_call("ls -lah / | head")

total 460K
drwxr-xr-x   1 root root 4.0K Oct  2 01:37 .
drwxr-xr-x   1 root root 4.0K Oct  2 01:37 ..
lrwxrwxrwx   1 root root    7 Jun 27  2024 bin -> usr/bin
drwxr-xr-x   2 root root 4.0K Apr 18  2022 boot
drwxr-xr-x   1 root root 4.0K Oct  2 02:30 content
-rw-r--r--   1 root root 4.3K Jul 10  2024 cuda-keyring_1.1-1_all.deb
drwxr-xr-x   1 root root 4.0K Sep 30 13:52 datalab
drwxr-xr-x   5 root root  360 Oct  2 01:37 dev
-rwxr-xr-x   1 root root    0 Oct  2 01:37 .dockerenv

In [ ]:
shell_call("apt-get install openjdk-8-jdk-headless -qq > /dev/null")

In [ ]:
# Fazendo download do SPARK
shell_call("wget -q https://dlcdn.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz")

In [ ]:
# Descompactando os arquivos
shell_call("tar xf /content/spark-3.5.7-bin-hadoop3.tgz")

In [ ]:
# Instalando Pacotes Necessários
#install.packages("googledrive")
#install.packages("sparklyr")
#install.packages("dplyr")
#install.packages("e1071")
#install.packages("ggplot2")

remotes::install_github("GabeChurch/sparkedatools", upgrade = "never")

Skipping install of 'sparkedatools' from a github remote, the SHA1 (b44fd0cc) has not changed since last install.
  Use `force = TRUE` to force installation



In [ ]:
library(sparklyr)
library(dplyr)
#' ---
library(e1071) # Para Skewness e Kurtosis
library(sparkedatools)
library(ggplot2)


Attaching package: ‘sparklyr’


The following object is masked from ‘package:stats’:

    filter



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘ggplot2’


The following object is masked from ‘package:e1071’:

    element




In [ ]:
Sys.setenv(SPARK_HOME="/content/spark-3.5.7-bin-hadoop3")

In [ ]:
spark_conn <- spark_connect(master = "local",
                   config=list(spark.sql.warehouse.dir=Sys.getenv("SPARK_HOME")))

In [ ]:
# ------------------------------------
# Baixando os dados
# ----
library(googledrive)
# ID do arquivo
file_id <- "14u4dUuTSYvTOIg4nWPJGai9f8VAXFavy"
destfile <- "base_e-commerce_parquet.zip"
# Baixar arquivo (publico ou com acesso)
drive_download(as_id(file_id), path = destfile, overwrite = TRUE)
# Descompactar zip
unzip(destfile)

The googledrive package is requesting access to your Google account.
Enter '1' to start a new auth process or select a pre-authorized account.
1: Send me to the browser for a new auth process.
2: jodavid.ferreira@ufpe.br


Selection: 2


Auto-refreshing stale OAuth token.

File downloaded:

• base_e-commerce_parquet.zip <id: 14u4dUuTSYvTOIg4nWPJGai9f8VAXFavy>

Saved locally as:

• base_e-commerce_parquet.zip



In [ ]:
#' -----------------------
#' Lendo arquivos parquet
#' -----------------------
dados <- spark_read_parquet(spark_conn, name = "base_e-commerce_parquet/database_compras_parquet.parquet" )


In [ ]:
#' -----------------------
#' Lista de data.frames existentes no Spark
#' -----------------------
src_tbls(spark_conn)

[1] "database_compras_parquet_6548bb65_d3bc_49f2_9ce7_053984b02cec"

In [ ]:
#' -----------------------
#' Verificando número de linhas
#' -----------------------
sdf_dim(dados)
sdf_nrow(dados)
sdf_ncol(dados)


[1] 338185     26

[1] 338185

[1] 26

In [ ]:
#' ------------------------------------
#' Verificando as colunas
#' -----------------
colnames(dados)
sdf_schema(dados)
#' -----------------

[1] "DsOrigem"           "DsCanalVenda"       "DsUnidadeNegocio"  
 [4] "DsTipo"             "NrPedido"           "DhPedido"          
 [7] "NrFilial"           "NrLojista"          "InMarketPlace"     
[10] "InRetira"           "VrVendaLiquida"     "VrDescontoTotal"   
[13] "VrFreteCliente"     "QtItem"             "NrItemLoja"        
[16] "NrItemSite"         "DsItemSite"         "NrDepartamentoSite"
[19] "DsDepartamentoSite" "NrSetorSite"        "DsSetorSite"       
[22] "NrFamiliaSite"      "DsFamiliaSite"      "NrMarcaSite"       
[25] "DsMarcaSite"        "DsSingleId"

$DsOrigem
$DsOrigem$name
[1] "DsOrigem"

$DsOrigem$type
[1] "StringType"


$DsCanalVenda
$DsCanalVenda$name
[1] "DsCanalVenda"

$DsCanalVenda$type
[1] "StringType"


$DsUnidadeNegocio
$DsUnidadeNegocio$name
[1] "DsUnidadeNegocio"

$DsUnidadeNegocio$type
[1] "StringType"


$DsTipo
$DsTipo$name
[1] "DsTipo"

$DsTipo$type
[1] "StringType"


$NrPedido
$NrPedido$name
[1] "NrPedido"

$NrPedido$type
[1] "IntegerType"


$DhPedido
$DhPedido$name
[1] "DhPedido"

$DhPedido$type
[1] "TimestampType"


$NrFilial
$NrFilial$name
[1] "NrFilial"

$NrFilial$type
[1] "IntegerType"


$NrLojista
$NrLojista$name
[1] "NrLojista"

$NrLojista$type
[1] "IntegerType"


$InMarketPlace
$InMarketPlace$name
[1] "InMarketPlace"

$InMarketPlace$type
[1] "BooleanType"


$InRetira
$InRetira$name
[1] "InRetira"

$InRetira$type
[1] "BooleanType"


$VrVendaLiquida
$VrVendaLiquida$name
[1] "VrVendaLiquida"

$VrVendaLiquida$type
[1] "DoubleType"


$VrDescontoTotal
$VrDescontoTotal$name
[1] "VrDescontoTotal"

$VrDescontoTotal$type
[1] "DoubleType"


$VrFreteCliente
$VrFreteCliente$name
[1] "VrFreteCliente"

$VrFreteCliente$type
[1] "DoubleType"


$QtItem
$QtItem$name
[1] "QtItem"

$QtItem$type
[1] "IntegerType"


$NrItemLoja
$NrItemLoja$name
[1] "NrItemLoja"

$NrItemLoja$type
[1] "IntegerType"


$NrItemSite
$NrItemSite$name
[1] "NrItemSite"

$NrItemSite$type
[1] "IntegerType"


$DsItemSite
$DsItemSite$name
[1] "DsItemSite"

$DsItemSite$type
[1] "StringType"


$NrDepartamentoSite
$NrDepartamentoSite$name
[1] "NrDepartamentoSite"

$NrDepartamentoSite$type
[1] "IntegerType"


$DsDepartamentoSite
$DsDepartamentoSite$name
[1] "DsDepartamentoSite"

$DsDepartamentoSite$type
[1] "StringType"


$NrSetorSite
$NrSetorSite$name
[1] "NrSetorSite"

$NrSetorSite$type
[1] "IntegerType"


$DsSetorSite
$DsSetorSite$name
[1] "DsSetorSite"

$DsSetorSite$type
[1] "StringType"


$NrFamiliaSite
$NrFamiliaSite$name
[1] "NrFamiliaSite"

$NrFamiliaSite$type
[1] "IntegerType"


$DsFamiliaSite
$DsFamiliaSite$name
[1] "DsFamiliaSite"

$DsFamiliaSite$type
[1] "StringType"


$NrMarcaSite
$NrMarcaSite$name
[1] "NrMarcaSite"

$NrMarcaSite$type
[1] "IntegerType"


$DsMarcaSite
$DsMarcaSite$name
[1] "DsMarcaSite"

$DsMarcaSite$type
[1] "StringType"


$DsSingleId
$DsSingleId$name
[1] "DsSingleId"

$DsSingleId$type
[1] "StringType"

### Dividir a base em treino e predição

In [ ]:
# Ordenando pela data

df_ordenado <- dados |>
  arrange(desc(DhPedido))

df_ordenado |>
  show()

# Source:     SQL [?? x 26]
# Database:   spark_connection
# Ordered by: desc(DhPedido)
   DsOrigem DsCanalVenda DsUnidadeNegocio DsTipo   NrPedido DhPedido           
   <chr>    <chr>        <chr>            <chr>       <int> <dttm>             
 1 SITE     APP          B2C              Produto 284950754 2021-08-30 23:56:00
 2 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 3 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 4 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 5 SITE     APP          B2C              Produto 284950249 2021-08-30 23:46:00
 6 SITE     SITE         B2C              Produto 284950255 2021-08-30 23:46:00
 7 SITE     APP          B2C              Produto 284950172 2021-08-30 23:44:00
 8 SITE     APP          B2C              Produto 284950166 2021-08-30 23:44:00
 9 SITE     APP          B2C              Produto 284950088 2021-08-30 23:42:00
10 SITE     APP          B2C    

In [ ]:
# Adicionar índice (numeração das linhas em Spark)
df_indexado <- df_ordenado |>
  sdf_with_sequential_id("linha_id")   # cria coluna linha_id (0-based)

In [ ]:
# Separando o data.frame dados em dois data.frames

# Separando as 100 linhas mais recentes
N = 100
dados_pred = df_indexado |>
  filter(`linha_id` <= N)

sdf_nrow(dados_pred)

# Criar DataFrame com o restante
dados_treino = df_indexado |>
  filter(`linha_id` > N)


sdf_nrow(dados_treino)

[1] 100

[1] 338085

In [ ]:
dados_pred |> show()

# Source:   SQL [?? x 27]
# Database: spark_connection
   DsOrigem DsCanalVenda DsUnidadeNegocio DsTipo   NrPedido DhPedido           
   <chr>    <chr>        <chr>            <chr>       <int> <dttm>             
 1 SITE     APP          B2C              Produto 284950754 2021-08-30 23:56:00
 2 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 3 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 4 SITE     MOBI         B2C              Produto 284950637 2021-08-30 23:54:00
 5 SITE     APP          B2C              Produto 284950249 2021-08-30 23:46:00
 6 SITE     SITE         B2C              Produto 284950255 2021-08-30 23:46:00
 7 SITE     APP          B2C              Produto 284950172 2021-08-30 23:44:00
 8 SITE     APP          B2C              Produto 284950166 2021-08-30 23:44:00
 9 SITE     APP          B2C              Produto 284950088 2021-08-30 23:42:00
10 SITE     APP          B2C              Produto 284950086 2021-

---
## Market Basket Analysis usando SparkLyr
---

In [ ]:
qtde_compras = dados_treino |>
  group_by(DsSingleId) |>
  count()

qtde_compras |>
  show()

# Source:   SQL [?? x 2]
# Database: spark_connection
# Groups:   DsSingleId
   DsSingleId                               n
   <chr>                                <dbl>
 1 a830ea66-fc16-11e9-b397-00163e63e134     4
 2 5c3a7c78-fc19-11e9-b06c-00163ef3deea     2
 3 a70a8672-fc18-11e9-87ef-00163ec5e1b7     1
 4 b14db6c4-fc16-11e9-afb0-00163e9515a7     1
 5 ab399954-fc18-11e9-9338-00163ec5e1b7     2
 6 a7a43678-fc18-11e9-b70e-00163ec5e1b7     1
 7 a4f2b6ca-fc18-11e9-b738-00163ef3deea     3
 8 5a451bee-fc19-11e9-a55a-00163ec5e1b7     4
 9 58556e92-fc19-11e9-af66-00163e9515a7     4
10 a75e0298-fc18-11e9-b0f5-00163e39813f     3
# ℹ more rows


### Implementação do FPGrowth

In [ ]:
df_grouped <- dados_treino |>
  group_by(DsSingleId) |>
  summarise(
    compras = collect_set(DsItemSite)  # mantém como array e deixa apenas um item de cada em cada lista
  )

df_grouped

# Source:   SQL [?? x 2]
# Database: spark_connection
   DsSingleId                           compras   
   <chr>                                <list>    
 1 0000bd90-218a-11eb-a1e5-00163e26abec <list [2]>
 2 0006482a-017f-11ec-aa9c-00163e457d21 <list [1]>
 3 0009ec12-218a-11eb-84c2-00163e89b637 <list [2]>
 4 000a3f20-017f-11ec-8861-00163e457d21 <list [1]>
 5 000ba644-c0e4-11ea-9f1d-00163e806a43 <list [1]>
 6 000c054e-44e7-11eb-b71b-00163ecac861 <list [2]>
 7 000d304c-9120-11eb-a629-00163e252137 <list [2]>
 8 000d78b8-2e16-11ea-ad93-00163ec8f72c <list [1]>
 9 00216f90-218a-11eb-a262-00163e7d45f4 <list [5]>
10 003c2636-9120-11eb-81a7-00163ef88abb <list [4]>
# ℹ more rows

In [ ]:
fpmodel <- df_grouped |>
  ml_fpgrowth(items_col = "compras",  min_support = 0.00005, min_confidence = 0.0001)

In [ ]:
print(ml_freq_itemsets(fpmodel))

# Source:   table<`sparklyr_tmp_8f91904d_ba80_4ce0_9cac_ffedff1ca47f`> [?? x 2]
# Database: spark_connection
   items       freq
   <list>     <dbl>
 1 <list [1]>    11
 2 <list [1]>    12
 3 <list [1]>    14
 4 <list [1]>     6
 5 <list [1]>     6
 6 <list [1]>    14
 7 <list [1]>    10
 8 <list [1]>    17
 9 <list [1]>     6
10 <list [1]>    12
# ℹ more rows


In [ ]:

print(ml_association_rules(fpmodel))

# Source:   table<`sparklyr_tmp_2c65410d_3dda_436d_bd52_63d6f2ee0795`> [?? x 5]
# Database: spark_connection
   antecedent consequent confidence   lift   support
   <list>     <list>          <dbl>  <dbl>     <dbl>
 1 <list [4]> <list [1]>     1       240.  0.0000909
 2 <list [4]> <list [1]>     1       221.  0.0000909
 3 <list [4]> <list [1]>     0.8     288.  0.0000727
 4 <list [4]> <list [1]>     0.7     379.  0.0000637
 5 <list [1]> <list [1]>     0.0395   14.5 0.0000546
 6 <list [4]> <list [1]>     0.429  1150.  0.0000818
 7 <list [4]> <list [1]>     0.476   831.  0.0000909
 8 <list [4]> <list [1]>     0.286   102.  0.0000546
 9 <list [4]> <list [1]>     0.571   137.  0.000109 
10 <list [4]> <list [1]>     0.952   251.  0.000182 
# ℹ more rows


### Vamos criar uma previsão com base nas regras de associação geradas

In [ ]:
df_pred_grouped <- dados_pred |>
  group_by(DsSingleId) |>
  summarise(
    compras = collect_set(DsItemSite)  # mantém como array e deixa apenas um item de cada em cada lista
  )


In [ ]:
df_pred_grouped

# Source:   SQL [?? x 2]
# Database: spark_connection
   DsSingleId                           compras   
   <chr>                                <list>    
 1 c59a2e50-7d7a-11eb-b030-00163e4a3f19 <list [1]>
 2 a518a8c6-fc18-11e9-abf3-00163e39813f <list [1]>
 3 58a48dd8-fc19-11e9-9cfb-00163ec5e1b7 <list [1]>
 4 abecf398-fc16-11e9-8486-00163e63e134 <list [1]>
 5 583f0a1c-fc19-11e9-af66-00163e9515a7 <list [1]>
 6 286f4d70-fc18-11e9-a8ea-00163e39813f <list [3]>
 7 6f567e9a-c989-11ea-9d26-00163eedb281 <list [1]>
 8 57cdcbea-fc19-11e9-a381-00163e63e134 <list [1]>
 9 56312688-fc19-11e9-8768-00163e63e134 <list [1]>
10 a6e7c83a-fc18-11e9-be05-00163ec5e1b7 <list [1]>
# ℹ more rows

In [ ]:
# prever usando o modelo acima
predicao <- fpmodel |>
  ml_transform(df_pred_grouped)

In [ ]:
predicao |>
  head()

# Source:   SQL [?? x 3]
# Database: spark_connection
  DsSingleId                           compras    prediction  
  <chr>                                <list>     <list>      
1 c59a2e50-7d7a-11eb-b030-00163e4a3f19 <list [1]> <list [0]>  
2 a518a8c6-fc18-11e9-abf3-00163e39813f <list [1]> <list [0]>  
3 58a48dd8-fc19-11e9-9cfb-00163ec5e1b7 <list [1]> <list [219]>
4 abecf398-fc16-11e9-8486-00163e63e134 <list [1]> <list [0]>  
5 583f0a1c-fc19-11e9-af66-00163e9515a7 <list [1]> <list [0]>  
6 286f4d70-fc18-11e9-a8ea-00163e39813f <list [3]> <list [4]>  

In [ ]:
pred_r <- predicao |> collect()

In [ ]:
head(pred_r$compras)

[[1]]
[[1]][[1]]
[1] "Cômoda Casa D Belize com 5 Gavetas"


[[2]]
[[2]][[1]]
[1] "Torneira Gourmet Prizi com Spray de Parede - KE6024"


[[3]]
[[3]][[1]]
[1] "Smart TV LED 50\\\" UHD 4K Samsung 50TU8000 Crystal UHD, Borda Infinita, Alexa Built In, Visual Livre de Cabos, Modo Ambiente Foto, Controle Único - 2020"


[[4]]
[[4]][[1]]
[1] "Caixa Amplificada Mondial CM-200 com Bluetooth, USB e Rádio FM - 200W"


[[5]]
[[5]][[1]]
[1] "Redmi Note 8 64gb + 4gb Ram – Space Balck (Preto)"


[[6]]
[[6]][[1]]
[1] "Refrigerador Brastemp Inverse BRE57AB Frost Free com Espaço Adapt 443L - Branco"

[[6]][[2]]
[1] "Cozinha Bartira Jaspe com 4 Portas, 3 Gavetas e 5 Prateleiras"

[[6]][[3]]
[1] "Balcão Duplo Bartira Jaspe com 2 Portas e 1 Prateleira"

In [ ]:
head(pred_r$prediction)

[[1]]
list()

[[2]]
list()

[[3]]
[[3]][[1]]
[1] "Smartphone Samsung Galaxy A32 Preto 128GB, 4GB RAM, Tela Infinita de 6.4\\\", Câmera Traseira Quádrupla, Bateria de 5000mAh, Dual Chip e Octa Core"

[[3]][[2]]
[1] "Guarda-Roupa Bartira Porto com 6 Portas e 2 Gavetas"

[[3]][[3]]
[1] "Armário Basculante Bartira Topázio com 2 Nichos"

[[3]][[4]]
[1] "Guarda-Roupa Bartira Ventura com 3 Portas e 4 Gavetas"

[[3]][[5]]
[1] "Base Box para Colchão de Casal Umaflex Fascinium 44x138x188 cm - Preto"

[[3]][[6]]
[1] "Balcão Triplo Bartira Rubi com 2 Portas e 3 Gavetas"

[[3]][[7]]
[1] "Cooktop a Gás Philco 5 Bocas Chef 5 Bisote Bivolt – Preto"

[[3]][[8]]
[1] "Smartphone Samsung Galaxy A51 Preto 128GB, Tela Infinita de 6.5\\\", Câmera Traseira Quádrupla, Leitor Digital na Tela, Android e Processador Octa-Core"

[[3]][[9]]
[1] "Liquidificador Mondial Power 2 Black L-28 - 350W"

[[3]][[10]]
[1] "Smart TV LED 55\\\" UHD 4K Samsung 55TU8000 Crystal UHD, Borda Infinita, Alexa Built In, Visual Livre de Cabos, Modo Ambiente Foto, Controle Único - 2020"

[[3]][[11]]
[1] "Guarda-Roupa Bartira Havana II com 7 Portas e 4 Gavetas"

[[3]][[12]]
[1] "Smartphone Samsung Galaxy A01 Vermelho 32GB, Tela Infinita de 5.7\\\", Câmera Traseira Dupla, Android 10.0, Dual Chip e Processador Octa-Core"

[[3]][[13]]
[1] "Smartphone Samsung Galaxy A01 Core Vermelho 32GB, Tela Infinita de 5.3”, Câmera Traseira 8MP, Android GO 10.0, Dual Chip e Processador Quad-Core"

[[3]][[14]]
[1] "Multifuncional Tanque de Tinta Epson EcoTank L3150 Wireless - Impressora, Copiadora, Scanner"

[[3]][[15]]
[1] "Refrigerador Brastemp Inverse BRE57AK Frost Free com Painel Eletrônico 443L - Evox"

[[3]][[16]]
[1] "Espremedor de Frutas Mondial Premium E-02 - Preto"

[[3]][[17]]
[1] "Conjunto de Facas Tramontina Plenus em Aço Inox e Polipropileno – 9 Peças"

[[3]][[18]]
[1] "Ferro a Vapor Electrolux Easyline SIE60 com Spray – Azul"

[[3]][[19]]
[1] "Armário Triplo Bartira Safira Plus com 3 Portas"

[[3]][[20]]
[1] "Smartphone Samsung Galaxy A10s Azul 32GB, Câmera Dupla Traseira, Selfie de 8MP, Tela Infinita de 6.2\\\", Leitor de Digital, Octa Core e Android 9.0"

[[3]][[21]]
[1] "Vivo Chip P60 4G"

[[3]][[22]]
[1] "Forno Elétrico Philco PFE48P com Função Timer Prata - 46L"

[[3]][[23]]
[1] "AOC Roku TV Smart TV LED 43” Full HD 43S5195/78 com Wi-fi, Controle Remoto com Atalhos, Roku Mobile, Miracast, Entradas HDMI e USB"

[[3]][[24]]
[1] "Smartphone Samsung Galaxy A31 Preto 128GB, 4GB RAM, Tela Infinita de 6.4\\\", Câmera Traseira Quádrupla, Leitor Digital na Tela e Android 10.0"

[[3]][[25]]
[1] "Refrigerador Brastemp BRM44HB Frost Free com Compartimento para Latas e Long Necks Branco - 375L"

[[3]][[26]]
[1] "Smartphone Samsung Galaxy A12 Branco 64GB, Tela Infinita de 6.5\\\", Câmera Quádrupla, Bateria 5000mAh, 4GB RAM e Processador Octa-Core"

[[3]][[27]]
[1] "Conjunto de Frigideiras Tramontina Turim com Revestimento Antiaderente - 3 Peças"

[[3]][[28]]
[1] "Smartphone Motorola Moto G9 Play Azul Safira 64GB, 4GB RAM, Tela de 6.5”, Câmera Traseira Tripla, Android 10 e Processador Octa-Core"

[[3]][[29]]
[1] "Fritadeira Sem Óleo Air Fryer Mondial Family NAF-03I 4L - Preto e Inox"

[[3]][[30]]
[1] "Colchão Casal Umaflex Itália com Pillow Top e Molas Ensacadas 26x138x188cm - Branco"

[[3]][[31]]
[1] "Lavadora de Roupas Consul 9Kg CWB09AB com Dosagem Extra Econômica - Branca"

[[3]][[32]]
[1] "Lavadora de Roupas Electrolux Automática LAC12 Topload com Dispenser Autolimpante e Cesto Inox 12kg - Branca"

[[3]][[33]]
[1] "Panela Elétrica de Arroz Mondial PE-43 6 Xícaras - Preto/Inox"

[[3]][[34]]
[1] "Smart TV LED 60\\\" UHD 4K LG 60UN7310PSA Wi-Fi, Bluetooth, HDR, Inteligência Artificial ThinQ AI, Google Assistente, Alexa, Controle Smart Magic - 2020"

[[3]][[35]]
[1] "Smartphone Samsung Galaxy A11 Preto 64GB, Câmera Tripla,Tela Infinita de 6.4\\\", Leitor de Digital, Octa Core, 3GB RAM, Carregamento Rápido e Android 10"

[[3]][[36]]
[1] "VM COND 9 000B PHILCO PAC9000TFM FRIO 220V"

[[3]][[37]